# Home Credit Default Risk – v4: loan-product encoding + LightGBM + CatBoost
Run every cell from top to bottom (**Runtime → Run all**). On Colab's free CPU this takes roughly **45–60 minutes** (CatBoost is the slow part). Keep the browser tab open while it runs.

**Validation results (5-fold CV AUC, same folds for every model):**

| Version | What changed | CV AUC |
|---|---|---|
| v1 | LightGBM, 52 features | 0.7613 |
| v2 | + CatBoost blend, 3 LightGBM seeds | 0.7629 |
| v3 | + Optuna-tuned LightGBM | 0.7632 |
| **v4** | **+ loan-product encoding (this notebook)** | **0.7694** |

| Model in this notebook | CV AUC |
|---|---|
| LightGBM (3 seeds) + target-encoded credit/annuity ratio + loan-structure score | 0.7676 |
| CatBoost + loan-product categories | 0.7681 |
| **Blend: 45% LightGBM + 55% CatBoost** | **0.7694** |

All random seeds are fixed so the results can be reproduced, as the rules require. Only `train.csv` and `test.csv` are used (no external data).

## The key idea in v4: the loan product tells you the bank's risk price
`AMT_CREDIT / AMT_ANNUITY` is roughly "how many payments" the loan has, and together with the interest rate it identifies the **loan product**.
The bank gives **higher interest rates to customers it considers riskier**, so the exact ratio carries the bank's own risk assessment.

For example, in the training data:

| credit ÷ annuity | customers | default rate |
|---|---|---|
| 20.00 (revolving loans) | 16,190 | 5.6% |
| 23.60 | 1,389 | 5.0% |
| 12.66 | 1,361 | 19.0% |
| 20.56 | 1,190 | 22.7% |

Tree models struggle to isolate such narrow exact values, so we add **target encoding**: each ratio value is replaced by the average default rate of training customers with the same value.

⚠️ Target encoding uses `TARGET`, so it must be computed **inside each CV fold, from the training part only**. Otherwise the model would see the answers of the rows it is evaluated on (leakage) and the CV score would be fake. The code below does this correctly. Training rows get their encoding from an inner 5-fold split, so no row ever sees its own label.

## Step 0: Setup

In [ ]:
# CatBoost is not pre-installed on Colab
!pip install -q catboost

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

In [ ]:
# Google Colab only
from google.colab import drive
drive.mount('/content/drive')

**Change this path** to the folder where you uploaded the `competition` folder.

In [ ]:
%cd "/content/drive/MyDrive/competition"

In [ ]:
current_dir = Path(os.getcwd())
train_file = current_dir / "input" / "train.csv"
test_file = current_dir / "input" / "test.csv"
sample_sub_file = current_dir / "input" / "sample_submission.csv"
print("Path OK" if train_file.exists() and test_file.exists() else "Path is WRONG – fix the %cd cell above")

## Step 1: Load the data
Train and test are joined into one table (`df`) so that every cleaning and feature step is applied to both in exactly the same way.

In [ ]:
train = pd.read_csv(train_file)
test = pd.read_csv(test_file)
sample_sub = pd.read_csv(sample_sub_file)

y = train["TARGET"].values
df = pd.concat([train.drop(columns="TARGET"), test], ignore_index=True)
n_train = len(train)
print(train.shape, test.shape, df.shape)

## Step 2: Clean the data
- `DAYS_EMPLOYED = 365243` is not a real value (about 1000 years). It mostly marks pensioners, so it becomes NaN and a flag column records it.
- `"XNA"` means "not available", so it becomes NaN.
- `DAYS_LAST_PHONE_CHANGE = 0` is most likely missing too.

In [ ]:
df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
df = df.replace("XNA", np.nan)
df.loc[df["DAYS_LAST_PHONE_CHANGE"] == 0, "DAYS_LAST_PHONE_CHANGE"] = np.nan

## Step 3: Create new features
Ratios often carry more signal than raw amounts, and the `EXT_SOURCE` external credit scores are the strongest predictors, so we combine them.

In [ ]:
# Money ratios
df["CREDIT_INCOME_RATIO"]  = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df["CREDIT_TERM"]          = df["AMT_ANNUITY"] / df["AMT_CREDIT"]      # roughly 1 / number of payments
df["CREDIT_GOODS_RATIO"]   = df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"]
df["CREDIT_GOODS_DIFF"]    = df["AMT_CREDIT"] - df["AMT_GOODS_PRICE"]
df["INCOME_PER_PERSON"]    = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]

# Time ratios
df["AGE_YEARS"]          = -df["DAYS_BIRTH"] / 365
df["EMPLOYED_AGE_RATIO"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
df["CAR_AGE_RATIO"]      = df["OWN_CAR_AGE"] / df["DAYS_BIRTH"]
df["PHONE_AGE_RATIO"]    = df["DAYS_LAST_PHONE_CHANGE"] / df["DAYS_BIRTH"]
df["ID_AGE_RATIO"]       = df["DAYS_ID_PUBLISH"] / df["DAYS_BIRTH"]

# External score combinations
ext = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
df["EXT_MEAN"]      = ext.mean(axis=1)
df["EXT_STD"]       = ext.std(axis=1)
df["EXT_MIN"]       = ext.min(axis=1)
df["EXT_MAX"]       = ext.max(axis=1)
df["EXT_PROD"]      = ext.prod(axis=1, min_count=3)
df["EXT_NAN_COUNT"] = ext.isnull().sum(axis=1)
df["EXT_WEIGHTED"]  = df["EXT_SOURCE_1"] * 2 + df["EXT_SOURCE_2"] * 1 + df["EXT_SOURCE_3"] * 3

# How much information is missing for this customer
df["N_MISSING"] = df.isnull().sum(axis=1)

## Step 4: Encode the categorical columns
LightGBM can use text columns directly once they are converted to the pandas `category` type.

In [ ]:
cat_cols = df.select_dtypes(exclude="number").columns.tolist()  # all text columns
for c in cat_cols:
    df[c] = df[c].astype("category")

features = [c for c in df.columns if c != "SK_ID_CURR"]
X = df.iloc[:n_train][features].reset_index(drop=True)
X_test = df.iloc[n_train:][features].reset_index(drop=True)
print("categorical:", cat_cols)
print("number of features:", len(features))

## Step 5: Loan-product keys
These are "ID-like" versions of the loan terms. They are **not** given to the models as plain numbers:
- LightGBM gets the **target-encoded** credit/annuity ratio (Step 6).
- CatBoost gets these keys as **categories** and does its own leak-safe target encoding ("ordered target statistics", Step 7).

In [ ]:
def sig_round(values, digits):
    """Round to a number of significant digits, e.g. 3 -> 0.0412 / 20.6 / 1230."""
    v = pd.Series(values, dtype=float)
    mag = np.floor(np.log10(np.abs(v).replace(0, np.nan))).fillna(0)
    return (np.round(v / 10 ** (mag - digits + 1)) * 10 ** (mag - digits + 1)).round(10)

credit, annuity, goods = df["AMT_CREDIT"], df["AMT_ANNUITY"], df["AMT_GOODS_PRICE"]
ratio = credit / annuity

# Keys for LightGBM target encoding
te_keys = pd.DataFrame({
    "RATIO_2DP": ratio.round(2),
    "RATIO_1DP": ratio.round(1),
})
te_keys_train = te_keys.iloc[:n_train].reset_index(drop=True)
te_keys_test = te_keys.iloc[n_train:].reset_index(drop=True)

# Inputs for the loan-structure score (Step 6): only the loan terms
loan = pd.DataFrame({
    "C": np.log(credit), "A": np.log(annuity), "G": np.log(goods), "R": ratio,
    "AG": annuity / goods, "CG": credit / goods,
    "TYPE": (df["NAME_CONTRACT_TYPE"] == "Cash loans").astype(float),
})
loan_train = loan.iloc[:n_train].reset_index(drop=True)
loan_test = loan.iloc[n_train:].reset_index(drop=True)

# Keys for CatBoost (used as categorical features)
cat_keys = pd.DataFrame({
    "RATIO_1DP_CAT": ratio.round(1),  # credit ÷ annuity, rounded to 1 decimal
    "RATIO_2DP_CAT": ratio.round(2),  # credit ÷ annuity, rounded to 2 decimals
    "AG_3SIG_CAT": sig_round(annuity / goods, 3),  # annuity ÷ goods price, 3 significant digits
    "ANNUITY_CAT": annuity,  # exact annuity amount
    "REGION_CAT": df["REGION_POPULATION_RELATIVE"].round(8),  # region (REGION_POPULATION_RELATIVE has only 81 values, so it is really a region ID)
})
cat_keys_train = cat_keys.iloc[:n_train].reset_index(drop=True)
cat_keys_test = cat_keys.iloc[n_train:].reset_index(drop=True)

# A quick look: default rate of the 10 most common ratio values
look = pd.DataFrame({"credit_annuity_ratio": te_keys_train["RATIO_2DP"], "TARGET": y})
look.groupby("credit_annuity_ratio")["TARGET"].agg(["count", "mean"]).sort_values("count", ascending=False).head(10)

## Step 6: LightGBM + target encoding, 5-fold CV, 3 seeds
For every fold:
1. The target encoding is built **from the training part of the fold only**:
   - training rows: from an inner 5-fold split (a row never uses its own label)
   - validation and test rows: from the whole training part
   - `M_SMOOTH = 30` blends rare ratio values towards the overall default rate, so a value seen only a few times does not get an extreme encoding
2. A **loan-structure score** is added: a small separate LightGBM that sees **only the loan terms** (credit, annuity, goods price, their ratios and the loan type) and predicts default. It learns the risk of each loan product, including combinations that rounding misses. Like the encoding, it is built from the training part only (inner 5-fold for training rows).
3. The main LightGBM is trained with the Optuna-tuned settings from v3, for 3 seeds, and averaged.

The **same 5 folds** are used for every model, so CV scores are comparable and the blend is fair.
On Colab this step takes roughly 20–25 minutes.

In [ ]:
M_SMOOTH = 30

def te_stats(keys, target):
    return pd.DataFrame({"k": keys, "y": target}).groupby("k")["y"].agg(["sum", "count"])

def te_apply(stats, keys, prior):
    s = pd.Series(keys).map(stats["sum"]).fillna(0).values
    c = pd.Series(keys).map(stats["count"]).fillna(0).values
    return (s + prior * M_SMOOTH) / (c + M_SMOOTH)

def add_target_encoding(tr_idx, va_idx):
    """Copies of the train / valid / test features with TE columns added, using ONLY the training fold's labels."""
    A_, B_, T_ = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy(), X_test.copy()
    for col in te_keys.columns:
        k_tr = te_keys_train[col].values
        prior = y[tr_idx].mean()
        inner = np.zeros(len(tr_idx))
        for itr, iva in KFold(5, shuffle=True, random_state=1).split(tr_idx):
            st = te_stats(k_tr[tr_idx[itr]], y[tr_idx[itr]])
            inner[iva] = te_apply(st, k_tr[tr_idx[iva]], y[tr_idx[itr]].mean())
        st = te_stats(k_tr[tr_idx], y[tr_idx])
        A_["TE_" + col] = inner
        B_["TE_" + col] = te_apply(st, k_tr[va_idx], prior)
        T_["TE_" + col] = te_apply(st, te_keys_test[col].values, prior)
    return A_, B_, T_


def loan_model():
    return lgb.LGBMClassifier(objective="binary", learning_rate=0.05, n_estimators=400, num_leaves=127,
                              min_child_samples=40, subsample=0.8, subsample_freq=1, reg_lambda=5.0,
                              verbose=-1, n_jobs=-1, random_state=0)


def add_loan_score(tr_idx, va_idx, A_, B_, T_):
    """Loan-structure score from a model that only sees the loan terms, built from the training fold only."""
    inner = np.zeros(len(tr_idx))
    for itr, iva in KFold(5, shuffle=True, random_state=1).split(tr_idx):
        m = loan_model().fit(loan_train.iloc[tr_idx[itr]], y[tr_idx[itr]])
        inner[iva] = m.predict_proba(loan_train.iloc[tr_idx[iva]])[:, 1]
    m = loan_model().fit(loan_train.iloc[tr_idx], y[tr_idx])
    A_["LOAN_SCORE"] = inner
    B_["LOAN_SCORE"] = m.predict_proba(loan_train.iloc[va_idx])[:, 1]
    T_["LOAN_SCORE"] = m.predict_proba(loan_test)[:, 1]

In [ ]:
# Optuna-tuned settings (from v3)
lgb_params = dict(
    objective="binary", metric="auc", learning_rate=0.02, n_estimators=20000,
    num_leaves=23, min_child_samples=125,
    subsample=0.9492640921811444, subsample_freq=1, colsample_bytree=0.24947959744878545,
    reg_lambda=60.16554740239231, reg_alpha=2.3419438263897017, min_split_gain=0.04966485832994068,
    cat_smooth=2.124799640518912, min_data_per_group=59, cat_l2=3.275036495773787,
    verbose=-1, n_jobs=-1,
)
LGB_SEEDS = [42, 7, 2024]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))

oof_lgb = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test))
oof_by_seed = {s: np.zeros(len(X)) for s in LGB_SEEDS}

for fold, (tr_idx, va_idx) in enumerate(folds):
    A_, B_, T_ = add_target_encoding(tr_idx, va_idx)
    add_loan_score(tr_idx, va_idx, A_, B_, T_)
    for seed in LGB_SEEDS:
        model = lgb.LGBMClassifier(**lgb_params, random_state=seed)
        model.fit(A_, y[tr_idx], eval_set=[(B_, y[va_idx])],
                  callbacks=[lgb.early_stopping(200, verbose=False)])
        p_va = model.predict_proba(B_)[:, 1]
        oof_by_seed[seed][va_idx] = p_va
        oof_lgb[va_idx] += p_va / len(LGB_SEEDS)
        pred_lgb += model.predict_proba(T_)[:, 1] / (len(folds) * len(LGB_SEEDS))
    print(f"Fold {fold}: AUC = {roc_auc_score(y[va_idx], oof_lgb[va_idx]):.5f}")

for s in LGB_SEEDS:
    print(f"Seed {s}: CV AUC = {roc_auc_score(y, oof_by_seed[s]):.5f}")
print(f"\nLightGBM (3 seeds) CV AUC = {roc_auc_score(y, oof_lgb):.5f}")

Which features did LightGBM rely on most? The new `TE_RATIO_*` and `LOAN_SCORE` features should be near the top.

In [ ]:
importance = pd.Series(model.feature_importances_, index=A_.columns).sort_values()
importance.tail(25).plot(kind="barh", figsize=(8, 8), title="LightGBM feature importance (last model)")

## Step 7: CatBoost + loan-product categories, 5-fold CV
CatBoost is especially good with categorical features. It target-encodes them itself with **ordered target statistics**, which is leak-safe by design.
Here it gets the normal features **plus the loan-product keys from Step 5 as categories**.

This is the slowest step: roughly 30–40 minutes on Colab's free CPU.

In [ ]:
X_cb = pd.concat([X, cat_keys_train], axis=1)
X_test_cb = pd.concat([X_test, cat_keys_test], axis=1)
cb_cats = cat_cols + list(cat_keys.columns)
for c in cb_cats:
    X_cb[c] = X_cb[c].astype(object).fillna("NA").astype(str)
    X_test_cb[c] = X_test_cb[c].astype(object).fillna("NA").astype(str)

oof_cat = np.zeros(len(X))
pred_cat = np.zeros(len(X_test))
test_pool = Pool(X_test_cb, cat_features=cb_cats)

for fold, (tr_idx, va_idx) in enumerate(folds):
    model_cb = CatBoostClassifier(
        iterations=10000, learning_rate=0.06, depth=6, l2_leaf_reg=5,
        eval_metric="AUC", random_seed=SEED, od_type="Iter", od_wait=300,
        verbose=0, thread_count=-1,
    )
    model_cb.fit(
        Pool(X_cb.iloc[tr_idx], y[tr_idx], cat_features=cb_cats),
        eval_set=Pool(X_cb.iloc[va_idx], y[va_idx], cat_features=cb_cats),
        use_best_model=True,
    )
    oof_cat[va_idx] = model_cb.predict_proba(X_cb.iloc[va_idx])[:, 1]
    pred_cat += model_cb.predict_proba(test_pool)[:, 1] / len(folds)
    print(f"Fold {fold}: AUC = {roc_auc_score(y[va_idx], oof_cat[va_idx]):.5f} (trees: {model_cb.get_best_iteration()})")

print(f"\nCatBoost CV AUC = {roc_auc_score(y, oof_cat):.5f}")

## Step 8: Blend the two models
A weighted average of the two models' predictions. The weight was chosen on the out-of-fold (CV) predictions: **45% LightGBM + 55% CatBoost**.

In [ ]:
for w in [0.0, 0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7, 1.0]:
    print(f"CatBoost weight {w:.2f}: CV AUC = {roc_auc_score(y, (1 - w) * oof_lgb + w * oof_cat):.5f}")

W_CAT = 0.55
oof_blend = (1 - W_CAT) * oof_lgb + W_CAT * oof_cat
pred = (1 - W_CAT) * pred_lgb + W_CAT * pred_cat
print(f"\nFinal blend CV AUC = {roc_auc_score(y, oof_blend):.5f}")

## Step 9: Save the submission (do not change this format)

In [ ]:
sample_sub["TARGET"] = pred
output_dir = current_dir / "output"
os.makedirs(output_dir, exist_ok=True)
sample_sub.to_csv(output_dir / "submission.csv", index=False)
print(sample_sub.shape)
sample_sub.head()

## Step 10: Save this notebook as a ZIP (required for Honors/Outstanding)
1. Save the notebook first (**Ctrl+S / Cmd+S**).
2. Run the cell below.
3. Download `my_model_v4.zip` and upload it on Omnicampus under **Competition (.zip)**.

In [ ]:
import zipfile
notebook_filename = "my_model_v4.ipynb"
zip_filename = "my_model_v4.zip"
notebook_path = current_dir / notebook_filename
if notebook_path.exists():
    with zipfile.ZipFile(current_dir / zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(notebook_path, arcname=notebook_filename)
    print("Created", current_dir / zip_filename)
else:
    print(f"Could not find {notebook_filename} in {current_dir}")